In [23]:
# ============================================================
# CELL 1 — IMPORTS & LOAD DATA
# ============================================================

import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import (
    PatternFill, Font, Alignment, Border, Side,
    GradientFill
)
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import (
    ColorScaleRule, DataBarRule, IconSetRule
)
from openpyxl.chart import BarChart, LineChart, Reference
from openpyxl.chart.series import DataPoint
import warnings
warnings.filterwarnings('ignore')

# Paths
processed_path = r'D:\Projects\End-to-end projects\12. Franchise Intelligence System\Data\Processed'
reports_path   = r'D:\Projects\End-to-end projects\12. Franchise Intelligence System\Reports'

# Load final ML output
outlet_scores = pd.read_csv(f'{processed_path}\\outlet_scores_final.csv')

print("✅ Libraries loaded")
print(f"outlet_scores shape: {outlet_scores.shape}")
print(f"\nClusters:\n{outlet_scores['cluster_label'].value_counts()}")
print(f"\nIntervention actions:\n{outlet_scores['intervention_action'].value_counts()}")

✅ Libraries loaded
outlet_scores shape: (85, 37)

Clusters:
cluster_label
Cluster B — Stable Operators     26
Cluster A — Star Performers      25
Cluster C — Needs Improvement    25
Cluster D — Critical Risk         9
Name: count, dtype: int64

Intervention actions:
intervention_action
LOW — Replicate Best Practices       38
HIGH — Action Plan Within 1 Month    22
MEDIUM — Monthly Review              16
IMMEDIATE — Exit or Restructure       9
Name: count, dtype: int64


In [24]:
# ============================================================
# CELL 2 — DEFINE EXCEL STYLES
# ============================================================

# ── Color Palette ───────────────────────────────────────────
DARK_BG        = '0F1117'
CARD_BG        = '1A1D27'
BORDER_COLOR   = '2E3250'
HEADER_BG      = '2E3250'
GREEN          = '00D4AA'
BLUE           = '4A90D9'
ORANGE         = 'F5A623'
RED            = 'E84A4A'
WHITE          = 'FFFFFF'
LIGHT_GRAY     = 'A0A0A0'
TEXT_COLOR     = 'E0E0E0'

# ── Fill Styles ─────────────────────────────────────────────
def make_fill(hex_color):
    return PatternFill(fill_type='solid',
                       fgColor=hex_color)

fill_dark      = make_fill(DARK_BG)
fill_card      = make_fill(CARD_BG)
fill_header    = make_fill(HEADER_BG)
fill_green     = make_fill(GREEN)
fill_blue      = make_fill(BLUE)
fill_orange    = make_fill(ORANGE)
fill_red       = make_fill(RED)
fill_white     = make_fill(WHITE)

# ── Font Styles ─────────────────────────────────────────────
def make_font(color=TEXT_COLOR, size=10,
              bold=False, italic=False):
    return Font(color=color, size=size,
                bold=bold, italic=italic,
                name='Segoe UI')

font_title     = make_font(GREEN,      24, bold=True)
font_subtitle  = make_font(LIGHT_GRAY, 11, italic=True)
font_header    = make_font(GREEN,      10, bold=True)
font_white_b   = make_font(WHITE,      10, bold=True)
font_white     = make_font(WHITE,      10)
font_gray      = make_font(LIGHT_GRAY,  9)
font_red_b     = make_font(RED,        10, bold=True)
font_orange_b  = make_font(ORANGE,     10, bold=True)
font_green_b   = make_font(GREEN,      10, bold=True)

# ── Border Styles ────────────────────────────────────────────
thin_border = Border(
    left   = Side(style='thin', color=BORDER_COLOR),
    right  = Side(style='thin', color=BORDER_COLOR),
    top    = Side(style='thin', color=BORDER_COLOR),
    bottom = Side(style='thin', color=BORDER_COLOR),
)

thick_border = Border(
    left   = Side(style='medium', color=GREEN),
    right  = Side(style='medium', color=GREEN),
    top    = Side(style='medium', color=GREEN),
    bottom = Side(style='medium', color=GREEN),
)

# ── Alignment ────────────────────────────────────────────────
align_center = Alignment(horizontal='center',
                          vertical='center',
                          wrap_text=True)
align_left   = Alignment(horizontal='left',
                          vertical='center',
                          wrap_text=True)
align_right  = Alignment(horizontal='right',
                          vertical='center')

# ── Helper: Apply background to entire row ───────────────────
def fill_row(ws, row, col_start, col_end, fill):
    for col in range(col_start, col_end + 1):
        ws.cell(row=row, column=col).fill = fill

# ── Helper: Style a cell completely ─────────────────────────
def style_cell(ws, row, col, value=None,
               fill=None, font=None,
               alignment=None, border=None,
               number_format=None):
    cell = ws.cell(row=row, column=col)
    if value is not None:
        cell.value = value
    if fill:
        cell.fill = fill
    if font:
        cell.font = font
    if alignment:
        cell.alignment = alignment
    if border:
        cell.border = border
    if number_format:
        cell.number_format = number_format
    return cell

# ── Intervention color mapping ───────────────────────────────
def get_intervention_fill(action):
    if 'IMMEDIATE' in str(action):
        return fill_red,   font_white_b
    elif 'HIGH' in str(action):
        return fill_orange, font_white_b
    elif 'MEDIUM' in str(action):
        return fill_blue,   font_white_b
    else:
        return fill_green,  make_font(DARK_BG, 10, bold=True)

# ── Cluster color mapping ────────────────────────────────────
def get_cluster_fill(label):
    if 'Star' in str(label):
        return fill_green,  make_font(DARK_BG, 10, bold=True)
    elif 'Stable' in str(label):
        return fill_blue,   font_white_b
    elif 'Improvement' in str(label):
        return fill_orange, font_white_b
    else:
        return fill_red,    font_white_b

print("✅ All styles defined successfully")
print(f"Total style objects created: fills, fonts, borders, helpers")

✅ All styles defined successfully
Total style objects created: fills, fonts, borders, helpers


In [25]:
# ============================================================
# CELL 3 — SHEET 1: EXECUTIVE SUMMARY
# ============================================================

wb = Workbook()
ws1 = wb.active
ws1.title = 'Executive Summary'

col_widths = {
    'A': 5,  'B': 18, 'C': 18, 'D': 18,
    'E': 18, 'F': 18, 'G': 18, 'H': 18,
    'I': 18, 'J': 18, 'K': 5
}
for col, width in col_widths.items():
    ws1.column_dimensions[col].width = width

for row in range(1, 60):
    fill_row(ws1, row, 1, 11, fill_dark)

# Title
ws1.merge_cells('B2:J2')
style_cell(ws1, 2, 2,
           value='FRANCHISE PERFORMANCE INTELLIGENCE SYSTEM',
           fill=fill_dark, font=font_title,
           alignment=align_center)

ws1.merge_cells('B3:J3')
style_cell(ws1, 3, 2,
           value='QSR India - 85 Outlets | 18 Months | Jan 2023 - Jun 2024',
           fill=fill_dark, font=font_subtitle,
           alignment=align_center)

ws1.row_dimensions[2].height = 40
ws1.row_dimensions[3].height = 20

ws1.merge_cells('B4:J4')
style_cell(ws1, 4, 2, value='', fill=make_fill(GREEN))
ws1.row_dimensions[4].height = 3

# KPI Cards Row 1
ws1.row_dimensions[6].height = 18
ws1.row_dimensions[7].height = 35

kpi_cards = [
    (2,  'TOTAL OUTLETS',       '85',         GREEN),
    (4,  'TOTAL REVENUE',       'Rs 32.8 Cr', GREEN),
    (6,  'AVG MONTHLY REV',     'Rs 2.15 Lakh', BLUE),
    (8,  'AVG RATING',          '3.84 / 5.0', BLUE),
    (10, 'TOTAL ORDERS',        '6,81,949',   GREEN),
]

for col_num, label, value, color in kpi_cards:
    end_col        = col_num + 1
    col_letter     = get_column_letter(col_num)
    end_col_letter = get_column_letter(end_col)

    ws1.merge_cells(f'{col_letter}6:{end_col_letter}6')
    style_cell(ws1, 6, col_num,
               value=label,
               fill=fill_card,
               font=make_font(LIGHT_GRAY, 9),
               alignment=align_center,
               border=thin_border)

    ws1.merge_cells(f'{col_letter}7:{end_col_letter}7')
    style_cell(ws1, 7, col_num,
               value=value,
               fill=fill_card,
               font=make_font(color, 16, bold=True),
               alignment=align_center,
               border=thin_border)

# KPI Cards Row 2
ws1.row_dimensions[9].height = 18
ws1.row_dimensions[10].height = 35

kpi_cards_2 = [
    (2,  'AVG NET MARGIN %',    '-10.9%',  RED),
    (4,  'AVG COMPLAINT RATE',  '11.5%',   ORANGE),
    (6,  'AVG REPEAT CUSTOMER', '48.5%',   GREEN),
    (8,  'CRITICAL OUTLETS',    '9',       RED),
    (10, 'AVG WASTE RATE',      '19.9%',   ORANGE),
]

for col_num, label, value, color in kpi_cards_2:
    end_col        = col_num + 1
    col_letter     = get_column_letter(col_num)
    end_col_letter = get_column_letter(end_col)

    ws1.merge_cells(f'{col_letter}9:{end_col_letter}9')
    style_cell(ws1, 9, col_num,
               value=label,
               fill=fill_card,
               font=make_font(LIGHT_GRAY, 9),
               alignment=align_center,
               border=thin_border)

    ws1.merge_cells(f'{col_letter}10:{end_col_letter}10')
    style_cell(ws1, 10, col_num,
               value=value,
               fill=fill_card,
               font=make_font(color, 16, bold=True),
               alignment=align_center,
               border=thin_border)

# Cluster Summary Table
ws1.row_dimensions[12].height = 20
ws1.merge_cells('B12:J12')
style_cell(ws1, 12, 2,
           value='CLUSTER PERFORMANCE SUMMARY',
           fill=fill_header,
           font=font_header,
           alignment=align_center)

headers     = ['Cluster', 'Outlets', 'Avg Score',
               'Avg Revenue', 'Net Margin %',
               'Avg Rating', 'Complaint %',
               'Waste %', 'Action Required']
header_cols = [2,3,4,5,6,7,8,9,10]

ws1.row_dimensions[13].height = 18
for col, header in zip(header_cols, headers):
    style_cell(ws1, 13, col,
               value=header,
               fill=fill_header,
               font=font_header,
               alignment=align_center,
               border=thin_border)

cluster_summary = outlet_scores.groupby('cluster_label').agg(
    outlets       = ('outlet_id',          'count'),
    avg_score     = ('composite_score',    'mean'),
    avg_revenue   = ('avg_monthly_revenue','mean'),
    avg_margin    = ('avg_net_margin_pct', 'mean'),
    avg_rating    = ('avg_rating',         'mean'),
    avg_complaint = ('avg_complaint_rate', 'mean'),
    avg_waste     = ('avg_waste_rate',     'mean'),
).round(1).reset_index()

cluster_order = [
    'Cluster A - Star Performers',
    'Cluster B - Stable Operators',
    'Cluster C - Needs Improvement',
    'Cluster D - Critical Risk',
]

action_map = {
    'Cluster A - Star Performers':   'Replicate Best Practices',
    'Cluster B - Stable Operators':  'Monthly Review',
    'Cluster C - Needs Improvement': 'Action Plan Within 1 Month',
    'Cluster D - Critical Risk':     'IMMEDIATE - Exit or Restructure',
}

# Fix cluster labels in outlet_scores to use - instead of -
outlet_scores['cluster_label'] = outlet_scores['cluster_label'].str.replace(' — ', ' - ')
outlet_scores['intervention_action'] = outlet_scores['intervention_action'].str.replace(' — ', ' - ')
outlet_scores['performance_tier'] = outlet_scores['performance_tier'].str.replace(' — ', ' - ')

# Recompute cluster summary after fix
cluster_summary = outlet_scores.groupby('cluster_label').agg(
    outlets       = ('outlet_id',          'count'),
    avg_score     = ('composite_score',    'mean'),
    avg_revenue   = ('avg_monthly_revenue','mean'),
    avg_margin    = ('avg_net_margin_pct', 'mean'),
    avg_rating    = ('avg_rating',         'mean'),
    avg_complaint = ('avg_complaint_rate', 'mean'),
    avg_waste     = ('avg_waste_rate',     'mean'),
).round(1).reset_index()

for i, cluster in enumerate(cluster_order):
    row      = 14 + i
    ws1.row_dimensions[row].height = 18
    row_data = cluster_summary[
        cluster_summary['cluster_label'] == cluster
    ]
    if len(row_data) == 0:
        continue
    r            = row_data.iloc[0]
    cfill, cfont = get_cluster_fill(cluster)

    values = [
        cluster.split('-')[1].strip(),
        int(r['outlets']),
        round(r['avg_score'], 1),
        f"Rs {r['avg_revenue']:,.0f}",
        f"{r['avg_margin']:.1f}%",
        round(r['avg_rating'], 2),
        f"{r['avg_complaint']:.1f}%",
        f"{r['avg_waste']:.1f}%",
        action_map[cluster],
    ]
    for col, val in zip(header_cols, values):
        style_cell(ws1, row, col,
                   value=val,
                   fill=cfill,
                   font=cfont,
                   alignment=align_center,
                   border=thin_border)

# Key Insights
ws1.row_dimensions[19].height = 20
ws1.merge_cells('B19:J19')
style_cell(ws1, 19, 2,
           value='KEY BUSINESS INSIGHTS',
           fill=fill_header,
           font=font_header,
           alignment=align_center)

insights = [
    ('\U0001f3c6', 'Star outlets generate Rs 3.46L/month avg revenue with 18.2% net margin - 4.7x better than critical outlets'),
    ('\U0001f6a8', '9 critical outlets averaging -91.5% net margin need immediate exit or restructure decision'),
    ('\U0001f4b8', 'Zomato & Swiggy commission costs Rs 5.59 Crore total - 24% of aggregator revenue lost to commissions'),
    ('\U0001f465', 'High staff turnover (>15%) correlates with Rs 2.1L lower monthly revenue and 0.95 lower rating'),
    ('\u23f0',     'Peak demand: Sunday 8 PM drives highest orders - critical outlets should staff up on weekends'),
    ('\U0001f3ea', 'Highway outlets best ROI at -0.2% avg margin vs Food Court at -16.4% - expansion priority clear'),
    ('\U0001f4c9', 'Waste rate of 30.9% in critical outlets vs 12.9% in star outlets - ops training needed urgently'),
    ('\U0001f504', 'Star outlets retain 56.5% repeat customers vs 35.1% in critical - loyalty gap is 21 percentage points'),
]

for i, (icon, insight) in enumerate(insights):
    row = 20 + i
    ws1.row_dimensions[row].height = 22
    style_cell(ws1, row, 2,
               value=icon,
               fill=fill_card,
               font=make_font(WHITE, 10, bold=True),
               alignment=align_center,
               border=thin_border)
    ws1.merge_cells(f'C{row}:J{row}')
    style_cell(ws1, row, 3,
               value=insight,
               fill=fill_card,
               font=make_font(TEXT_COLOR, 10),
               alignment=align_left,
               border=thin_border)

# Footer
ws1.row_dimensions[29].height = 15
ws1.merge_cells('B29:J29')
style_cell(ws1, 29, 2,
           value='Franchise Intelligence System | Python | PostgreSQL | Scikit-learn | Power BI | Jan 2023 - Jun 2024',
           fill=fill_dark,
           font=make_font(LIGHT_GRAY, 8, italic=True),
           alignment=align_center)

print("✅ Sheet 1 - Executive Summary built successfully")

✅ Sheet 1 - Executive Summary built successfully


In [26]:
# ============================================================
# CELL 4 — SHEET 2: FRANCHISE SCORECARD
# ============================================================

ws2 = wb.create_sheet(title='Franchise Scorecard')

col_widths_2 = {
    'A': 4,  'B': 10, 'C': 14, 'D': 14,
    'E': 14, 'F': 16, 'G': 14, 'H': 14,
    'I': 14, 'J': 13, 'K': 13, 'L': 13,
    'M': 13, 'N': 13, 'O': 13, 'P': 14,
    'Q': 28, 'R': 4
}
for col, width in col_widths_2.items():
    ws2.column_dimensions[col].width = width

for row in range(1, 95):
    fill_row(ws2, row, 1, 18, fill_dark)

# Title
ws2.merge_cells('B2:Q2')
style_cell(ws2, 2, 2,
           value='FRANCHISE PERFORMANCE SCORECARD - ALL 85 OUTLETS',
           fill=fill_dark, font=font_title,
           alignment=align_center)
ws2.row_dimensions[2].height = 35

ws2.merge_cells('B3:Q3')
style_cell(ws2, 3, 2,
           value='Sorted by Composite Score | Color coded by Intervention Priority',
           fill=fill_dark, font=font_subtitle,
           alignment=align_center)
ws2.row_dimensions[3].height = 18

ws2.merge_cells('B4:Q4')
style_cell(ws2, 4, 2, value='', fill=make_fill(GREEN))
ws2.row_dimensions[4].height = 3

# Column Headers
headers_2 = [
    'Rank', 'Outlet ID', 'City', 'Archetype',
    'Cluster', 'Composite\nScore', 'Monthly\nRevenue Rs',
    'Net Margin\n%', 'Avg\nRating', 'Complaint\nRate %',
    'Repeat\nCust %', 'Waste\nRate %', 'Staff\nTurnover %',
    'Upsell\nRate %', 'Avg Order\nValue Rs',
    'Performance\nTier', 'Intervention Action'
]
header_start_col = 2
ws2.row_dimensions[5].height = 30

for i, header in enumerate(headers_2):
    col = header_start_col + i
    style_cell(ws2, 5, col,
               value=header,
               fill=fill_header,
               font=font_header,
               alignment=align_center,
               border=thin_border)

# Data
scorecard_cols = [
    'performance_rank',
    'outlet_id',
    'city',
    'archetype',
    'cluster_label',
    'composite_score',
    'avg_monthly_revenue',
    'avg_net_margin_pct',
    'avg_rating',
    'avg_complaint_rate',
    'avg_repeat_pct',
    'avg_waste_rate',
    'avg_staff_turnover',
    'avg_upsell_rate',
    'avg_order_value',
    'performance_tier',
    'intervention_action',
]

scorecard_data = outlet_scores[scorecard_cols].sort_values(
    'composite_score', ascending=False
).reset_index(drop=True)

for i, row_data in scorecard_data.iterrows():
    row     = 6 + i
    ws2.row_dimensions[row].height = 16

    action       = row_data['intervention_action']
    ifill, ifont = get_intervention_fill(action)
    row_fill     = fill_card if i % 2 == 0 else fill_dark

    for j, col_name in enumerate(scorecard_cols):
        col   = header_start_col + j
        value = row_data[col_name]

        # Format values
        if col_name == 'avg_monthly_revenue':
            value = f"Rs {value:,.0f}"
        elif col_name in ['avg_net_margin_pct', 'avg_complaint_rate',
                          'avg_repeat_pct', 'avg_waste_rate',
                          'avg_staff_turnover', 'avg_upsell_rate']:
            value = f"{value:.1f}%"
        elif col_name == 'composite_score':
            value = round(float(value), 1)
        elif col_name == 'avg_rating':
            value = round(float(value), 2)
        elif col_name == 'avg_order_value':
            value = f"Rs {value:,.0f}"
        elif col_name == 'cluster_label':
            value = value.split('-')[1].strip() \
                    if '-' in str(value) else value
        elif col_name == 'performance_tier':
            value = value.split('-')[1].strip() \
                    if '-' in str(value) else value

        # Apply cell styles
        if col_name == 'intervention_action':
            style_cell(ws2, row, col,
                       value=value,
                       fill=ifill,
                       font=ifont,
                       alignment=align_center,
                       border=thin_border)
        elif col_name == 'cluster_label':
            cfill, cfont = get_cluster_fill(row_data['cluster_label'])
            style_cell(ws2, row, col,
                       value=value,
                       fill=cfill,
                       font=cfont,
                       alignment=align_center,
                       border=thin_border)
        elif col_name == 'composite_score':
            score = float(row_data['composite_score'])
            if score >= 65:
                score_fill = make_fill(GREEN)
                score_font = make_font(DARK_BG, 10, bold=True)
            elif score >= 45:
                score_fill = make_fill(BLUE)
                score_font = font_white_b
            elif score >= 30:
                score_fill = make_fill(ORANGE)
                score_font = font_white_b
            else:
                score_fill = make_fill(RED)
                score_font = font_white_b
            style_cell(ws2, row, col,
                       value=value,
                       fill=score_fill,
                       font=score_font,
                       alignment=align_center,
                       border=thin_border)
        elif col_name == 'avg_net_margin_pct':
            margin = float(row_data['avg_net_margin_pct'])
            mfill  = make_fill(RED)    if margin < 0 else \
                     make_fill(ORANGE) if margin < 5 else \
                     make_fill(GREEN)
            mfont  = font_white_b if margin < 5 else \
                     make_font(DARK_BG, 10, bold=True)
            style_cell(ws2, row, col,
                       value=f"{margin:.1f}%",
                       fill=mfill,
                       font=mfont,
                       alignment=align_center,
                       border=thin_border)
        else:
            style_cell(ws2, row, col,
                       value=value,
                       fill=row_fill,
                       font=font_white,
                       alignment=align_center,
                       border=thin_border)

print(f"✅ Sheet 2 - Franchise Scorecard built")
print(f"Total outlet rows written: {len(scorecard_data)}")

✅ Sheet 2 - Franchise Scorecard built
Total outlet rows written: 85


In [27]:
# ============================================================
# CELL 5 — SHEET 3: INTERVENTION ACTION PLAN
# ============================================================

ws3 = wb.create_sheet(title='Intervention Action Plan')

col_widths_3 = {
    'A': 4,  'B': 10, 'C': 14, 'D': 14,
    'E': 14, 'F': 14, 'G': 14, 'H': 14,
    'I': 14, 'J': 14, 'K': 30, 'L': 30,
    'M': 4
}
for col, width in col_widths_3.items():
    ws3.column_dimensions[col].width = width

for row in range(1, 120):
    fill_row(ws3, row, 1, 13, fill_dark)

# Title
ws3.merge_cells('B2:L2')
style_cell(ws3, 2, 2,
           value='FRANCHISE INTERVENTION PRIORITY ACTION PLAN',
           fill=fill_dark, font=font_title,
           alignment=align_center)
ws3.row_dimensions[2].height = 35

ws3.merge_cells('B3:L3')
style_cell(ws3, 3, 2,
           value='Prioritized by Risk Score | Actionable recommendations per outlet tier',
           fill=fill_dark, font=font_subtitle,
           alignment=align_center)
ws3.row_dimensions[3].height = 18

ws3.merge_cells('B4:L4')
style_cell(ws3, 4, 2, value='', fill=make_fill(RED))
ws3.row_dimensions[4].height = 3

# Intervention sections
intervention_sections = [
    (
        'IMMEDIATE - Exit or Restructure',
        RED,
        'IMMEDIATE ACTION REQUIRED',
        'These outlets have composite score < 30 and net margin < -20%. '
        'Immediate ops review, cost restructure or franchise exit recommended.',
    ),
    (
        'HIGH - Action Plan Within 1 Month',
        ORANGE,
        'HIGH RISK - ACTION WITHIN 1 MONTH',
        'These outlets show declining trends across multiple KPIs. '
        'Assign regional manager support and build 30-day recovery plan.',
    ),
    (
        'MEDIUM - Monthly Review',
        BLUE,
        'MEDIUM RISK - MONTHLY REVIEW',
        'These outlets are stable but showing early warning signals. '
        'Include in monthly ops review and track KPI trends closely.',
    ),
    (
        'LOW - Replicate Best Practices',
        GREEN,
        'LOW RISK - REPLICATE BEST PRACTICES',
        'Top performing outlets. Document their operational SOPs, '
        'staff training protocols and menu mix for replication across network.',
    ),
]

col_headers = [
    'Outlet ID', 'City', 'Archetype',
    'Score', 'Net Margin %', 'Rating',
    'Complaint %', 'Waste %',
    'Staff Turnover %', 'Mgr Changes'
]

current_row = 6

for action, color, section_title, description in intervention_sections:
    section_fill = make_fill(color)
    section_data = outlet_scores[
        outlet_scores['intervention_action'] == action
    ].sort_values('composite_score').reset_index(drop=True)

    if len(section_data) == 0:
        continue

    # Section header
    ws3.merge_cells(f'B{current_row}:K{current_row}')
    style_cell(ws3, current_row, 2,
               value=section_title,
               fill=section_fill,
               font=make_font(
                   DARK_BG if color == GREEN else WHITE,
                   13, bold=True
               ),
               alignment=align_center)
    ws3.row_dimensions[current_row].height = 25
    current_row += 1

    # Description
    ws3.merge_cells(f'B{current_row}:K{current_row}')
    style_cell(ws3, current_row, 2,
               value=description,
               fill=fill_card,
               font=make_font(LIGHT_GRAY, 9, italic=True),
               alignment=align_left)
    ws3.row_dimensions[current_row].height = 20
    current_row += 1

    # Column headers
    for j, header in enumerate(col_headers):
        style_cell(ws3, current_row, 2 + j,
                   value=header,
                   fill=make_fill(HEADER_BG),
                   font=make_font(color, 9, bold=True),
                   alignment=align_center,
                   border=thin_border)
    ws3.row_dimensions[current_row].height = 18
    current_row += 1

    # Data rows
    for idx, row_data in section_data.iterrows():
        ws3.row_dimensions[current_row].height = 16
        row_bg = fill_card if idx % 2 == 0 else fill_dark
        values = [
            row_data['outlet_id'],
            row_data['city'],
            row_data['archetype'],
            round(float(row_data['composite_score']), 1),
            f"{row_data['avg_net_margin_pct']:.1f}%",
            round(float(row_data['avg_rating']), 2),
            f"{row_data['avg_complaint_rate']:.1f}%",
            f"{row_data['avg_waste_rate']:.1f}%",
            f"{row_data['avg_staff_turnover']:.1f}%",
            int(row_data['total_manager_changes']),
        ]
        for j, val in enumerate(values):
            style_cell(ws3, current_row, 2 + j,
                       value=val,
                       fill=row_bg,
                       font=font_white,
                       alignment=align_center,
                       border=thin_border)
        current_row += 1

    ws3.row_dimensions[current_row].height = 10
    current_row += 2

print(f"✅ Sheet 3 - Intervention Action Plan built")
print(f"Total rows used: ~{current_row}")

✅ Sheet 3 - Intervention Action Plan built
Total rows used: ~111


In [28]:
# ============================================================
# CELL 6 — SHEET 4: CITY & CLUSTER ANALYSIS
# ============================================================

ws4 = wb.create_sheet(title='City & Cluster Analysis')

col_widths_4 = {
    'A': 4,  'B': 16, 'C': 8,  'D': 16,
    'E': 14, 'F': 14, 'G': 14, 'H': 14,
    'I': 14, 'J': 14, 'K': 14, 'L': 4
}
for col, width in col_widths_4.items():
    ws4.column_dimensions[col].width = width

for row in range(1, 80):
    fill_row(ws4, row, 1, 12, fill_dark)

# Title
ws4.merge_cells('B2:K2')
style_cell(ws4, 2, 2,
           value='CITY & CLUSTER DEEP DIVE ANALYSIS',
           fill=fill_dark, font=font_title,
           alignment=align_center)
ws4.row_dimensions[2].height = 35

ws4.merge_cells('B3:K3')
style_cell(ws4, 3, 2,
           value='Performance benchmarks by city tier and ML cluster',
           fill=fill_dark, font=font_subtitle,
           alignment=align_center)
ws4.row_dimensions[3].height = 18

ws4.merge_cells('B4:K4')
style_cell(ws4, 4, 2, value='', fill=make_fill(BLUE))
ws4.row_dimensions[4].height = 3

# City table
ws4.merge_cells('B6:K6')
style_cell(ws4, 6, 2,
           value='CITY-LEVEL PERFORMANCE BENCHMARKS',
           fill=fill_header,
           font=font_header,
           alignment=align_center)
ws4.row_dimensions[6].height = 20

city_headers = [
    'City', 'Tier', 'Outlets',
    'Avg Monthly\nRevenue Rs', 'Net Margin\n%',
    'Avg Rating', 'Complaint\n%',
    'Repeat\nCust %', 'Waste\n%',
    'Revenue/Rent\nRatio'
]
ws4.row_dimensions[7].height = 28
for j, header in enumerate(city_headers):
    style_cell(ws4, 7, 2 + j,
               value=header,
               fill=fill_header,
               font=font_header,
               alignment=align_center,
               border=thin_border)

city_data = outlet_scores.groupby('city').agg(
    outlets       = ('outlet_id',          'count'),
    tier          = ('tier',               'first'),
    avg_revenue   = ('avg_monthly_revenue','mean'),
    avg_margin    = ('avg_net_margin_pct', 'mean'),
    avg_rating    = ('avg_rating',         'mean'),
    avg_complaint = ('avg_complaint_rate', 'mean'),
    avg_repeat    = ('avg_repeat_pct',     'mean'),
    avg_waste     = ('avg_waste_rate',     'mean'),
).round(2).reset_index()
city_data = city_data.sort_values(
    ['tier', 'avg_revenue'], ascending=[True, False]
)

tier_color_map = {1: GREEN, 2: BLUE, 3: ORANGE}

for i, (_, row_data) in enumerate(city_data.iterrows()):
    row    = 8 + i
    ws4.row_dimensions[row].height = 16
    tier   = int(row_data['tier'])
    tcolor = tier_color_map.get(tier, WHITE)
    row_bg = fill_card if i % 2 == 0 else fill_dark
    rental = {1: 120000, 2: 75000, 3: 45000}
    rev_rent_ratio = round(row_data['avg_revenue'] / rental[tier], 2)

    values = [
        row_data['city'],
        f"Tier {tier}",
        int(row_data['outlets']),
        f"Rs {row_data['avg_revenue']:,.0f}",
        f"{row_data['avg_margin']:.1f}%",
        round(row_data['avg_rating'], 2),
        f"{row_data['avg_complaint']:.1f}%",
        f"{row_data['avg_repeat']:.1f}%",
        f"{row_data['avg_waste']:.1f}%",
        rev_rent_ratio,
    ]
    for j, val in enumerate(values):
        if j == 1:
            style_cell(ws4, row, 2 + j,
                       value=val,
                       fill=make_fill(tcolor),
                       font=make_font(
                           DARK_BG if tcolor == GREEN else WHITE,
                           9, bold=True
                       ),
                       alignment=align_center,
                       border=thin_border)
        else:
            style_cell(ws4, row, 2 + j,
                       value=val,
                       fill=row_bg,
                       font=font_white,
                       alignment=align_center,
                       border=thin_border)

# Cluster deep dive table
cluster_start = 8 + len(city_data) + 3

ws4.merge_cells(f'B{cluster_start}:K{cluster_start}')
style_cell(ws4, cluster_start, 2,
           value='ML CLUSTER DEEP DIVE - KPI BENCHMARKS',
           fill=fill_header,
           font=font_header,
           alignment=align_center)
ws4.row_dimensions[cluster_start].height = 20

cluster_headers = [
    'Cluster', 'Outlets', 'Avg Score',
    'Avg Revenue Rs', 'Net Margin %',
    'Avg Rating', 'Complaint %',
    'Waste %', 'Staff Turnover %',
    'Training Hrs'
]
ws4.row_dimensions[cluster_start + 1].height = 18
for j, header in enumerate(cluster_headers):
    style_cell(ws4, cluster_start + 1, 2 + j,
               value=header,
               fill=fill_header,
               font=font_header,
               alignment=align_center,
               border=thin_border)

cluster_deep = outlet_scores.groupby('cluster_label').agg(
    outlets      = ('outlet_id',          'count'),
    avg_score    = ('composite_score',    'mean'),
    avg_revenue  = ('avg_monthly_revenue','mean'),
    avg_margin   = ('avg_net_margin_pct', 'mean'),
    avg_rating   = ('avg_rating',         'mean'),
    avg_complaint= ('avg_complaint_rate', 'mean'),
    avg_waste    = ('avg_waste_rate',     'mean'),
    avg_turnover = ('avg_staff_turnover', 'mean'),
    avg_training = ('avg_training_hours', 'mean'),
).round(2).reset_index()

for i, cluster in enumerate(cluster_order):
    row      = cluster_start + 2 + i
    ws4.row_dimensions[row].height = 18
    row_data = cluster_deep[
        cluster_deep['cluster_label'] == cluster
    ]
    if len(row_data) == 0:
        continue
    r            = row_data.iloc[0]
    cfill, cfont = get_cluster_fill(cluster)
    values = [
        cluster.split('-')[1].strip(),
        int(r['outlets']),
        round(r['avg_score'], 1),
        f"Rs {r['avg_revenue']:,.0f}",
        f"{r['avg_margin']:.1f}%",
        round(r['avg_rating'], 2),
        f"{r['avg_complaint']:.1f}%",
        f"{r['avg_waste']:.1f}%",
        f"{r['avg_turnover']:.1f}%",
        round(r['avg_training'], 1),
    ]
    for j, val in enumerate(values):
        style_cell(ws4, row, 2 + j,
                   value=val,
                   fill=cfill,
                   font=cfont,
                   alignment=align_center,
                   border=thin_border)

print(f"✅ Sheet 4 - City & Cluster Analysis built")

✅ Sheet 4 - City & Cluster Analysis built


In [29]:
# ============================================================
# CELL 7 — SAVE EXCEL FILE
# ============================================================

excel_path = f'{reports_path}\\Franchise_Intelligence_Scorecard.xlsx'
wb.save(excel_path)

print(f"✅ Excel scorecard saved successfully")
print(f"\nFile : Franchise_Intelligence_Scorecard.xlsx")
print(f"Path : {reports_path}")
print(f"\nSheets created:")
for sheet in wb.sheetnames:
    print(f"  ✅ {sheet}")

✅ Excel scorecard saved successfully

File : Franchise_Intelligence_Scorecard.xlsx
Path : D:\Projects\End-to-end projects\12. Franchise Intelligence System\Reports

Sheets created:
  ✅ Executive Summary
  ✅ Franchise Scorecard
  ✅ Intervention Action Plan
  ✅ City & Cluster Analysis
